In [1]:
import os
os.chdir(r"C:\Users\Prana\OneDrive\Documents\GitHub\infosys-langgraph-email-assistant-group2")
print(os.getcwd())

C:\Users\Prana\OneDrive\Documents\GitHub\infosys-langgraph-email-assistant-group2


In [2]:
import pandas as pd
df=pd.read_csv("data/sample_emails_with_triage_200.csv")
df.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,respond,neutral
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,notify,urgent
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,respond,neutral
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,neutral


In [3]:
def email_assistant(body):
    text = body.lower()
    if "invoice" in text:
        return "respond", "urgent"
    if "meeting" in text:
        return "respond", "polite"
    if "internship" in text:
        return "notify", "polite"
    if "query" in text:
        return "respond", "neutral"
    if "newsletter" in text:
        return "ignore", "neutral"
    return "ignore", "neutral"

In [4]:
# Define dangerous actions, HITL checkpoint and human approval
dangerous_actions = ["respond"]

def hitl_check(action):
    return "Wait_for_human" if action in dangerous_actions else "Auto_approve"

def human_decision():
    decision = input("Approve action? (yes/no): ")
    return decision.lower() == "yes"

In [5]:
results = []
for _, row in df.sample(5).iterrows():
    action, tone = email_assistant(row['body'])
    status = hitl_check(action)
    if status == "Wait_for_human":
        approved = human_decision()
        final_action = action if approved else "blocked"
    else:
        final_action = action
    results.append({
        "email": row['body'],
        "ai_action": action,
        "final_action": final_action,
        "hitl_status": status
    })
results_df = pd.DataFrame(results)
results_df

,email,ai_action,final_action,hitl_status
0,Security alert: multiple failed login attempts...,ignore,ignore,Auto_approve
1,Your order #6464 has been shipped and is expec...,ignore,ignore,Auto_approve
2,Security alert: multiple failed login attempts...,ignore,ignore,Auto_approve
3,Notice: Your account will be locked unless ver...,ignore,ignore,Auto_approve
4,Congratulations! You have been selected as a l...,ignore,ignore,Auto_approve
